In [1]:
# %% [markdown]
# # 🧬 Simulador ODE — Red metabólica: producción, degradación y dimerización
# Basado en **ODEs-2.pdf (págs. 1–22)**. Caso de estudio "Proteínas en Acción":
#
# - **R1** producción de novo de X: `+Xpre·kprod·Eprod` → constante `a`
# - **R2** degradación de X: `−X·kdeg·Edeg` → `−b·X`
# - **R3+** dimerización: `−X·Y·kdim·Edim` → `−c·X·Y`
# - **R3−** disociación: `+XY·kdis·Edis` → `+d·XY`
#
# Sistema completo (3D):
# ```
# dX/dt  = a − b·X − c·X·Y + d·XY
# dY/dt  = −c·X·Y + d·XY
# dXY/dt = +c·X·Y − d·XY
# ```
# **Ecuación de conservación:** `Y(t) + XY(t) = Ytotal` → reducción 3D → 2D.

# %%
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from ipywidgets import interact, FloatSlider, Checkbox

# %%
def sistema_3D(y, t, a, b, c, d, e, degrada_Y):
    X, Y, XY = y
    r_dimer = c * X * Y      # R3+
    r_disoc = d * XY         # R3-
    dX  = a - b * X - r_dimer + r_disoc
    dY  = -r_dimer + r_disoc - (e * Y if degrada_Y else 0.0)  # R4 opcional
    dXY = r_dimer - r_disoc
    return [dX, dY, dXY]

def sistema_2D(y, t, a, b, c, d, Ytotal):
    """Versión reducida usando la conservación: XY = Ytotal − Y."""
    X, Y = y
    XY = Ytotal - Y
    dX = a - b * X - c * X * Y + d * XY
    dY = -c * X * Y + d * XY
    return [dX, dY]

def estado_estacionario(a, b, c, d, Ytotal):
    """Paso 5 de la receta: derivadas = 0 (sin R4)."""
    Xs = a / b
    q  = c / d
    Ys = Ytotal / (1 + q * Xs)
    return Xs, Ys, Ytotal - Ys

# %% [markdown]
# ## 1) Widget: red completa 3D + extensión R4 (degradación de Y, "el colaborador")

# %%
def explorar_red(X0, Y0, Ytotal, a, b, c, d, degrada_Y, e, t_max):
    XY0 = max(0.0, Ytotal - Y0)
    t = np.linspace(0, t_max, 1000)
    sol = odeint(sistema_3D, [X0, Y0, XY0], t, args=(a, b, c, d, e, degrada_Y))

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 7), sharex=True,
                                   gridspec_kw={'height_ratios': [3, 1]})
    ax1.plot(t, sol[:, 0], lw=2.4, color='crimson', label='X(t)')
    ax1.plot(t, sol[:, 1], lw=2.4, color='dodgerblue', label='Y(t)')
    ax1.plot(t, sol[:, 2], lw=2.4, color='goldenrod', label='XY(t)')
    if not degrada_Y:
        Xs, Ys, XYs = estado_estacionario(a, b, c, d, Ytotal)
        ax1.axhline(Xs, ls='--', c='crimson', alpha=.4, label=f'X*={Xs:.2f}')
        ax1.axhline(Ys, ls='--', c='dodgerblue', alpha=.4, label=f'Y*={Ys:.2f}')
        ax1.axhline(XYs, ls='--', c='goldenrod', alpha=.4, label=f'XY*={XYs:.2f}')
    ax1.set_ylabel('Concentración'); ax1.grid(alpha=.3)
    ax1.set_title('Red metabólica: R1 producción · R2 degradación · R3± dimerización',
                  fontweight='bold')
    ax1.legend(loc='center left', fontsize=8, ncol=3)

    ax2.plot(t, sol[:, 1] + sol[:, 2], lw=2.2,
             color='red' if degrada_Y else 'seagreen')
    ax2.axhline(Ytotal, ls='--', c='gray', alpha=.7, label=f'Ytotal={Ytotal}')
    ax2.set_xlabel('Tiempo t'); ax2.set_ylabel('Y + XY')
    ax2.grid(alpha=.3); ax2.legend(fontsize=8)
    plt.tight_layout(); plt.show()

    err = np.max(np.abs(sol[:, 1] + sol[:, 2] - Ytotal))
    if degrada_Y:
        print(f'R4 activa: la conservación SE ROMPE (desviación máx = {err:.3f})')
        print('d(Y+XY)/dt = −e·Y  →  el pool de Y se agota y todo termina en X* = a/b')
    else:
        print(f'✓ Conservación verificada: error máximo |Y+XY−Ytotal| = {err:.2e}')
        Xs, Ys, XYs = estado_estacionario(a, b, c, d, Ytotal)
        print(f'Estado estacionario analítico: X*={Xs:.3f}, Y*={Ys:.3f}, XY*={XYs:.3f}')

interact(explorar_red,
         X0=FloatSlider(value=0, min=0, max=20, step=0.5, description='X₀:'),
         Y0=FloatSlider(value=5, min=0, max=20, step=0.5, description='Y₀:'),
         Ytotal=FloatSlider(value=5, min=0.5, max=20, step=0.5, description='Ytotal:'),
         a=FloatSlider(value=2.0, min=0, max=10, step=0.1, description='a=Xpre·kprod·Eprod:'),
         b=FloatSlider(value=0.3, min=0, max=2, step=0.05, description='b=kdeg·Edeg:'),
         c=FloatSlider(value=0.5, min=0, max=2, step=0.05, description='c=kdim·Edim:'),
         d=FloatSlider(value=0.4, min=0.05, max=2, step=0.05, description='d=kdis·Edis:'),
         degrada_Y=Checkbox(value=False, description='R4: degradar Y'),
         e=FloatSlider(value=0.2, min=0, max=2, step=0.05, description='e=kdegY·EdegY:'),
         t_max=FloatSlider(value=50, min=5, max=200, step=5, description='t máx:'))

# %% [markdown]
# ## 2) Reducción 3D → 2D con la ecuación de conservación (Paso 3: Simplificar)
# Como `Y + XY = Ytotal` es constante, `XY(t) = Ytotal − Y(t)` y nos ahorramos una ecuación diferencial entera.

# %%
def comparar_reduccion(Y0, Ytotal, a, b, c, d, t_max):
    X0, XY0 = 0.0, max(0.0, Ytotal - Y0)
    t = np.linspace(0, t_max, 800)
    sol3 = odeint(sistema_3D, [X0, Y0, XY0], t, args=(a, b, c, d, 0.0, False))
    sol2 = odeint(sistema_2D, [X0, Y0], t, args=(a, b, c, d, Ytotal))

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(t, sol3[:, 0], lw=2.4, color='crimson', label='X — sistema 3D')
    ax.plot(t, sol2[:, 0], 'o', ms=3, color='crimson', alpha=.5, label='X — sistema 2D')
    ax.plot(t, sol3[:, 1], lw=2.4, color='dodgerblue', label='Y — sistema 3D')
    ax.plot(t, sol2[:, 1], 'o', ms=3, color='dodgerblue', alpha=.5, label='Y — sistema 2D')
    ax.plot(t, sol3[:, 2], lw=2.4, color='goldenrod', label='XY — sistema 3D')
    ax.plot(t, Ytotal - sol2[:, 1], 'o', ms=3, color='goldenrod', alpha=.5,
            label='XY = Ytotal−Y (reconstruido del 2D)')
    ax.set_title('Equivalencia 3D ↔ 2D por conservación', fontweight='bold')
    ax.set_xlabel('Tiempo t'); ax.set_ylabel('Concentración')
    ax.grid(alpha=.3); ax.legend(fontsize=8)
    plt.show()

    print('Diferencia máxima entre sistemas:')
    print(f'  |X₃ᴰ − X₂ᴰ|  = {np.max(np.abs(sol3[:,0]-sol2[:,0])):.2e}')
    print(f'  |Y₃ᴰ − Y₂ᴰ|  = {np.max(np.abs(sol3[:,1]-sol2[:,1])):.2e}')
    print(f'  |XY₃ᴰ − (Ytotal−Y₂ᴰ)| = {np.max(np.abs(sol3[:,2]-(Ytotal-sol2[:,1]))):.2e}')

interact(comparar_reduccion,
         Y0=FloatSlider(value=5, min=0, max=20, step=0.5, description='Y₀:'),
         Ytotal=FloatSlider(value=5, min=0.5, max=20, step=0.5, description='Ytotal:'),
         a=FloatSlider(value=2.0, min=0, max=10, step=0.1, description='a:'),
         b=FloatSlider(value=0.3, min=0.05, max=2, step=0.05, description='b:'),
         c=FloatSlider(value=0.5, min=0, max=2, step=0.05, description='c:'),
         d=FloatSlider(value=0.4, min=0.05, max=2, step=0.05, description='d:'),
         t_max=FloatSlider(value=50, min=5, max=200, step=5, description='t máx:'))

# %% [markdown]
# ## 3) Paso 5 de la receta: verificar el estado estacionario (derivadas = 0)

# %%
a, b, c, d, Ytotal = 2.0, 0.3, 0.5, 0.4, 5.0
Xs, Ys, XYs = estado_estacionario(a, b, c, d, Ytotal)
print(f'Analítico:  X* = a/b = {Xs:.4f}')
print(f'            Y* = Ytotal/(1+(c/d)·X*) = {Ys:.4f}')
print(f'            XY* = Ytotal − Y* = {XYs:.4f}')

# 1) ¿Las derivadas se anulan en el estado estacionario?
f_ss = sistema_3D([Xs, Ys, XYs], 0, a, b, c, d, 0.0, False)
print('\nDerivadas evaluadas en (X*,Y*,XY*):', [f'{v:.2e}' for v in f_ss])

# 2) ¿La simulación larga converge ahí?
t = np.linspace(0, 300, 1000)
fin = odeint(sistema_3D, [0, Ytotal, 0], t, args=(a, b, c, d, 0.0, False))[-1]
print(f'Numérico (t→∞):  X={fin[0]:.4f}, Y={fin[1]:.4f}, XY={fin[2]:.4f}')
print('El sistema tiene UN ÚNICO comportamiento estructural, pero múltiples',
      'resultados según sus parámetros (Paso 4 del PVI).')

interactive(children=(FloatSlider(value=0.0, description='X₀:', max=20.0, step=0.5), FloatSlider(value=5.0, de…

interactive(children=(FloatSlider(value=5.0, description='Y₀:', max=20.0, step=0.5), FloatSlider(value=5.0, de…

Analítico:  X* = a/b = 6.6667
            Y* = Ytotal/(1+(c/d)·X*) = 0.5357
            XY* = Ytotal − Y* = 4.4643

Derivadas evaluadas en (X*,Y*,XY*): ['0.00e+00', '0.00e+00', '0.00e+00']
Numérico (t→∞):  X=6.6667, Y=0.5357, XY=4.4643
El sistema tiene UN ÚNICO comportamiento estructural, pero múltiples resultados según sus parámetros (Paso 4 del PVI).
